# 01 · Multi-Paradigm Retrieval & Controls — *SycophancyRAGBench*

Compares Sparse (BM25), Dense (BGE), and Hybrid (RRF) retrieval alongside Oracle Evidence and Irrelevant Distractor controls.

In [ ]:
import sys, os
pkg_root = os.path.abspath("..") if os.path.exists("../configs") else os.path.abspath(".")
if pkg_root not in sys.path:
    sys.path.insert(0, pkg_root)

from configs import config as cfg
from src import env, utils as U
from src.data_loader import load_or_build_benchmark
from src.retrieval.manager import run_retrieval_pipeline

env.setup_workspace()
claims, prompts = load_or_build_benchmark()
print(f"Loaded {len(claims)} claims.")


In [ ]:
# Run all retrieval paradigms & experimental controls
retrievers = ["bm25", "dense", "hybrid", "oracle", "irrelevant"]
results_paths = run_retrieval_pipeline(claims, retrievers_to_run=retrievers, k=cfg.TOP_K)


In [ ]:
# Inspect comparative retrieval quality (Recall@k & MRR@10)
metrics = U.load_json(cfg.RETRIEVAL_METRICS)
import pandas as pd
table = []
for m, vals in metrics.items():
    if "recall_at_k" in vals:
        row = {"Retriever": m.upper(), "MRR@10": vals.get("mrr_at_10")}
        row.update(vals["recall_at_k"])
        table.append(row)

df_retr = pd.DataFrame(table)
print("=== RETRIEVAL COMPARISON (FEVER GOLD EVIDENCE) ===")
print(df_retr.to_string(index=False))
